In [ ]:
!pip install scanpy anndata scikit-misc torch_geometric

In [ ]:
# !pip install scanpy anndata scikit-misc

# ------------------------------ Model CODE ----------------------

import os
import scipy
import anndata
import sklearn
import torch
import random
import numpy as np
import scanpy as sc
import pandas as pd
from typing import Optional
import scipy.sparse as sp
from torch.backends import cudnn
from scipy.sparse import coo_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.neighbors import kneighbors_graph

def construct_neighbor_graph(adata_omics1, adata_omics2, datatype='SPOTS', n_neighbors=3):
    """
    Construct neighbor graphs, including feature graph and spatial graph.
    Feature graph is based expression data while spatial graph is based on cell/spot spatial coordinates.

    Parameters
    ----------
    n_neighbors : int
        Number of neighbors.

    Returns
    -------
    data : dict
        AnnData objects with preprossed data for different omics.

    """

    # construct spatial neighbor graphs
    ################# spatial graph #################
    if datatype in ['Stereo-CITE-seq', 'Spatial-epigenome-transcriptome']:
       n_neighbors=6
    # omics1
    cell_position_omics1 = adata_omics1.obsm['spatial']
    adj_omics1 = construct_graph_by_coordinate(cell_position_omics1, n_neighbors=n_neighbors)
    adata_omics1.uns['adj_spatial'] = adj_omics1

    # omics2
    cell_position_omics2 = adata_omics2.obsm['spatial']
    adj_omics2 = construct_graph_by_coordinate(cell_position_omics2, n_neighbors=n_neighbors)
    adata_omics2.uns['adj_spatial'] = adj_omics2

    ################# feature graph #################
    feature_graph_omics1, feature_graph_omics2 = construct_graph_by_feature(adata_omics1, adata_omics2)
    adata_omics1.obsm['adj_feature'], adata_omics2.obsm['adj_feature'] = feature_graph_omics1, feature_graph_omics2

    data = {'adata_omics1': adata_omics1, 'adata_omics2': adata_omics2}

    return data

def pca(adata, use_reps=None, n_comps=10):

    """Dimension reduction with PCA algorithm"""

    from sklearn.decomposition import PCA
    from scipy.sparse.csc import csc_matrix
    from scipy.sparse.csr import csr_matrix
    pca = PCA(n_components=n_comps)
    if use_reps is not None:
       feat_pca = pca.fit_transform(adata.obsm[use_reps])
    else:
       if isinstance(adata.X, csc_matrix) or isinstance(adata.X, csr_matrix):
          feat_pca = pca.fit_transform(adata.X.toarray())
       else:
          feat_pca = pca.fit_transform(adata.X)

    return feat_pca

def clr_normalize_each_cell(adata, inplace=True):

    """Normalize count vector for each cell, i.e. for each row of .X"""

    import numpy as np
    import scipy

    def seurat_clr(x):
        # TODO: support sparseness
        s = np.sum(np.log1p(x[x > 0]))
        exp = np.exp(s / len(x))
        return np.log1p(x / exp)

    if not inplace:
        adata = adata.copy()

    # apply to dense or sparse matrix, along axis. returns dense matrix
    adata.X = np.apply_along_axis(
        seurat_clr, 1, (adata.X.toarray() if scipy.sparse.issparse(adata.X) else np.array(adata.X))
    )
    return adata

def construct_graph_by_feature(adata_omics1, adata_omics2, k=20, mode= "connectivity", metric="correlation", include_self=False):

    """Constructing feature neighbor graph according to expresss profiles"""

    feature_graph_omics1=kneighbors_graph(adata_omics1.obsm['feat'], k, mode=mode, metric=metric, include_self=include_self)
    feature_graph_omics2=kneighbors_graph(adata_omics2.obsm['feat'], k, mode=mode, metric=metric, include_self=include_self)

    return feature_graph_omics1, feature_graph_omics2

def construct_graph_by_coordinate(cell_position, n_neighbors=3):
    #print('n_neighbor:', n_neighbors)
    """Constructing spatial neighbor graph according to spatial coordinates."""

    nbrs = NearestNeighbors(n_neighbors=n_neighbors+1).fit(cell_position)
    _ , indices = nbrs.kneighbors(cell_position)
    x = indices[:, 0].repeat(n_neighbors)
    y = indices[:, 1:].flatten()
    adj = pd.DataFrame(columns=['x', 'y', 'value'])
    adj['x'] = x
    adj['y'] = y
    adj['value'] = np.ones(x.size)
    return adj

def transform_adjacent_matrix(adjacent):
    n_spot = adjacent['x'].max() + 1
    adj = coo_matrix((adjacent['value'], (adjacent['x'], adjacent['y'])), shape=(n_spot, n_spot))
    return adj

def sparse_mx_to_torch_sparse_tensor(sparse_mx):

    """Convert a scipy sparse matrix to a torch sparse tensor."""

    sparse_mx = sparse_mx.tocoo().astype(np.float32)
    indices = torch.from_numpy(np.vstack((sparse_mx.row, sparse_mx.col)).astype(np.int64))
    values = torch.from_numpy(sparse_mx.data)
    shape = torch.Size(sparse_mx.shape)
    return torch.sparse.FloatTensor(indices, values, shape)

# ====== Graph preprocessing
def preprocess_graph(adj):
    adj = sp.coo_matrix(adj)
    adj_ = adj + sp.eye(adj.shape[0])
    rowsum = np.array(adj_.sum(1))
    degree_mat_inv_sqrt = sp.diags(np.power(rowsum, -0.5).flatten())
    adj_normalized = adj_.dot(degree_mat_inv_sqrt).transpose().dot(degree_mat_inv_sqrt).tocoo()
    return sparse_mx_to_torch_sparse_tensor(adj_normalized)

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors, kneighbors_graph
from scipy.spatial.distance import cdist

def build_dual_graph(adata_omics1, adata_omics2, device='cpu', num_neighbors=15):
    """Constructing ARISE dual graphs (sim, dist, common) as PyTorch Geometric edge_index format"""

    RNA_expression = adata_omics1.obsm['feat']
    if hasattr(RNA_expression, 'toarray'):
        RNA_expression = RNA_expression.toarray()
    cell_positions = adata_omics1.obsm['spatial']

    # 1. Similarity Graph (RNA)
    similarity_matrix = cosine_similarity(RNA_expression)
    nbrs = NearestNeighbors(n_neighbors=num_neighbors + 1, metric='cosine').fit(RNA_expression)
    distances, indices = nbrs.kneighbors(RNA_expression)

    adjacency_matrix = np.zeros_like(similarity_matrix, dtype=int)
    for i in range(len(RNA_expression)):
        for j in indices[i][1:]:  # Skip self
            adjacency_matrix[i, j] = 1
            adjacency_matrix[j, i] = 1  # Make graph undirected

    sim_edge_index = torch.tensor(np.array(np.nonzero(adjacency_matrix)), dtype=torch.long).to(device)
    sim_edge_weight = torch.tensor(similarity_matrix[adjacency_matrix > 0], dtype=torch.float).to(device)

    # 2. Distance Graph (Spatial)
    distance_matrix = cdist(cell_positions, cell_positions, metric='euclidean')
    knn_graph = kneighbors_graph(cell_positions, n_neighbors=num_neighbors, mode='distance', include_self=False)
    knn_graph = knn_graph.maximum(knn_graph.T)  # Symmetrize physical graph

    dist_edge_index = torch.tensor(knn_graph.nonzero(), dtype=torch.long).to(device)
    dist_edge_weight = torch.tensor(knn_graph.data, dtype=torch.float).to(device)

    # 3. Common Graph (Intersection)
    sim_edges = set(zip(sim_edge_index[0].tolist(), sim_edge_index[1].tolist()))
    dist_edges = set(zip(dist_edge_index[0].tolist(), dist_edge_index[1].tolist()))
    common_edges = sim_edges.intersection(dist_edges)

    if len(common_edges) > 0:
        common_edge_index = torch.tensor(list(zip(*common_edges)), dtype=torch.long).to(device)
    else:
        common_edge_index = torch.empty((2, 0), dtype=torch.long).to(device)
    common_edge_weight = torch.ones(common_edge_index.shape[1], dtype=torch.float).to(device)

    adj = {
        'sim_edge_index': sim_edge_index,
        'sim_edge_weight': sim_edge_weight,
        'dist_edge_index': dist_edge_index,
        'dist_edge_weight': dist_edge_weight,
        'common_edge_index': common_edge_index,
        'common_edge_weight': common_edge_weight
    }

    return adj

def adjacent_matrix_preprocessing(adata_omics1, adata_omics2):
    """Converting dense adjacent matrix to sparse adjacent matrix"""

    ######################################## construct spatial graph ########################################
    adj_spatial_omics1 = adata_omics1.uns['adj_spatial']
    adj_spatial_omics1 = transform_adjacent_matrix(adj_spatial_omics1)
    adj_spatial_omics2 = adata_omics2.uns['adj_spatial']
    adj_spatial_omics2 = transform_adjacent_matrix(adj_spatial_omics2)

    adj_spatial_omics1 = adj_spatial_omics1.toarray()   # To ensure that adjacent matrix is symmetric
    adj_spatial_omics2 = adj_spatial_omics2.toarray()

    adj_spatial_omics1 = adj_spatial_omics1 + adj_spatial_omics1.T
    adj_spatial_omics1 = np.where(adj_spatial_omics1>1, 1, adj_spatial_omics1)
    adj_spatial_omics2 = adj_spatial_omics2 + adj_spatial_omics2.T
    adj_spatial_omics2 = np.where(adj_spatial_omics2>1, 1, adj_spatial_omics2)

    # convert dense matrix to sparse matrix
    adj_spatial_omics1 = preprocess_graph(adj_spatial_omics1) # sparse adjacent matrix corresponding to spatial graph
    adj_spatial_omics2 = preprocess_graph(adj_spatial_omics2)

    ######################################## construct feature graph ########################################
    adj_feature_omics1 = torch.FloatTensor(adata_omics1.obsm['adj_feature'].copy().toarray())
    adj_feature_omics2 = torch.FloatTensor(adata_omics2.obsm['adj_feature'].copy().toarray())

    adj_feature_omics1 = adj_feature_omics1 + adj_feature_omics1.T
    adj_feature_omics1 = np.where(adj_feature_omics1>1, 1, adj_feature_omics1)
    adj_feature_omics2 = adj_feature_omics2 + adj_feature_omics2.T
    adj_feature_omics2 = np.where(adj_feature_omics2>1, 1, adj_feature_omics2)

    # convert dense matrix to sparse matrix
    adj_feature_omics1 = preprocess_graph(adj_feature_omics1) # sparse adjacent matrix corresponding to feature graph
    adj_feature_omics2 = preprocess_graph(adj_feature_omics2)

    adj = {'adj_spatial_omics1': adj_spatial_omics1,
           'adj_spatial_omics2': adj_spatial_omics2,
           'adj_feature_omics1': adj_feature_omics1,
           'adj_feature_omics2': adj_feature_omics2,
           }

    return adj

def lsi(
        adata: anndata.AnnData, n_components: int = 20,
        use_highly_variable: Optional[bool] = None, **kwargs
       ) -> None:
    r"""
    LSI analysis (following the Seurat v3 approach)
    """
    if use_highly_variable is None:
        use_highly_variable = "highly_variable" in adata.var
    adata_use = adata[:, adata.var["highly_variable"]] if use_highly_variable else adata
    X = tfidf(adata_use.X)
    #X = adata_use.X
    X_norm = sklearn.preprocessing.Normalizer(norm="l1").fit_transform(X)
    X_norm = np.log1p(X_norm * 1e4)
    X_lsi = sklearn.utils.extmath.randomized_svd(X_norm, n_components, **kwargs)[0]
    X_lsi -= X_lsi.mean(axis=1, keepdims=True)
    X_lsi /= X_lsi.std(axis=1, ddof=1, keepdims=True)
    #adata.obsm["X_lsi"] = X_lsi
    adata.obsm["X_lsi"] = X_lsi[:,1:]

def tfidf(X):
    r"""
    TF-IDF normalization (following the Seurat v3 approach)
    """
    idf = X.shape[0] / X.sum(axis=0)
    if scipy.sparse.issparse(X):
        tf = X.multiply(1 / X.sum(axis=1))
        return tf.multiply(idf)
    else:
        tf = X / X.sum(axis=1, keepdims=True)
        return tf * idf

def fix_seed(seed):
    #seed = 2023
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = True
    cudnn.benchmark = False

    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
from torch.nn.modules.module import Module


from torch_geometric.nn import GCNConv



# ---- scModFuse Modular Fusion Classes ----
# ---- scModFuse & SpatialGLue Modular Fusion Classes ----
class AttentionLayer(Module):
    """Standard Softmax Attention Layer."""
    def __init__(self, in_feat, out_feat, dropout=0.0, act=F.relu):
        super(AttentionLayer, self).__init__()
        self.in_feat = in_feat
        self.out_feat = out_feat
        self.w_omega = Parameter(torch.FloatTensor(in_feat, out_feat))
        self.u_omega = Parameter(torch.FloatTensor(out_feat, 1))
        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.xavier_uniform_(self.w_omega)
        torch.nn.init.xavier_uniform_(self.u_omega)

    def forward(self, emb1, emb2):
        emb = []
        emb.append(torch.unsqueeze(torch.squeeze(emb1), dim=1))
        emb.append(torch.unsqueeze(torch.squeeze(emb2), dim=1))
        self.emb = torch.cat(emb, dim=1)
        self.v = torch.tanh(torch.matmul(self.emb, self.w_omega))
        self.vu = torch.matmul(self.v, self.u_omega)
        self.alpha = F.softmax(torch.squeeze(self.vu) + 1e-6, dim=-1)
        emb_combined = torch.matmul(torch.transpose(self.emb, 1, 2), torch.unsqueeze(self.alpha, -1))
        return torch.squeeze(emb_combined), self.alpha

class GatedFusionLayer(nn.Module):
    """Gated Modality Fusion Layer."""
    def __init__(self, dim):
        super(GatedFusionLayer, self).__init__()
        self.fc_s = nn.Linear(dim, dim, bias=True)
        self.fc_f = nn.Linear(dim, dim, bias=True)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.fc_s.weight)
        nn.init.xavier_uniform_(self.fc_f.weight)
        nn.init.zeros_(self.fc_s.bias)
        nn.init.zeros_(self.fc_f.bias)

    def forward(self, emb_s, emb_f):
        gate = torch.sigmoid(self.fc_s(emb_s) + self.fc_f(emb_f))
        fused = gate * emb_s + (1.0 - gate) * emb_f
        g = gate.mean(dim=-1, keepdim=True)
        alpha = torch.cat([g, 1.0 - g], dim=-1)
        return fused, alpha

class QKVCrossFusionLayer(nn.Module):
    """QKV Cross-Modality Attention Fusion Layer."""
    def __init__(self, dim, attention_type='local'):
        super().__init__()
        self.dim = dim
        self.attention_type = attention_type
        self.scale = dim ** -0.5
        self.q_proj1 = nn.Linear(dim, dim, bias=False)
        self.k_proj1 = nn.Linear(dim, dim, bias=False)
        self.v_proj1 = nn.Linear(dim, dim, bias=False)
        self.q_proj2 = nn.Linear(dim, dim, bias=False)
        self.k_proj2 = nn.Linear(dim, dim, bias=False)
        self.v_proj2 = nn.Linear(dim, dim, bias=False)
        self.fc_out = nn.Linear(2 * dim, dim, bias=True)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.q_proj1.weight)
        nn.init.xavier_uniform_(self.k_proj1.weight)
        nn.init.xavier_uniform_(self.v_proj1.weight)
        nn.init.xavier_uniform_(self.q_proj2.weight)
        nn.init.xavier_uniform_(self.k_proj2.weight)
        nn.init.xavier_uniform_(self.v_proj2.weight)
        nn.init.xavier_uniform_(self.fc_out.weight)
        nn.init.zeros_(self.fc_out.bias)

    def forward(self, emb1, emb2):
        q1, k1, v1 = self.q_proj1(emb1), self.k_proj1(emb1), self.v_proj1(emb1)
        q2, k2, v2 = self.q_proj2(emb2), self.k_proj2(emb2), self.v_proj2(emb2)
        if self.attention_type == 'global':
            attn_scores1 = torch.matmul(q2, k1.T) * self.scale
            z1 = torch.matmul(F.softmax(attn_scores1, dim=-1), v1)
            attn_scores2 = torch.matmul(q1, k2.T) * self.scale
            z2 = torch.matmul(F.softmax(attn_scores2, dim=-1), v2)
            alpha = torch.cat([F.softmax(attn_scores1, dim=-1).mean(dim=-1, keepdim=True), F.softmax(attn_scores2, dim=-1).mean(dim=-1, keepdim=True)], dim=-1)
        else:
            w1 = torch.sigmoid((q2 * k1).sum(dim=-1, keepdim=True) * self.scale)
            w2 = torch.sigmoid((q1 * k2).sum(dim=-1, keepdim=True) * self.scale)
            z1, z2 = w1 * v1, w2 * v2
            alpha = torch.cat([w1, w2], dim=-1)
        return self.fc_out(torch.cat([z1, z2], dim=-1)), alpha

class HierarchicalFusionLayer(nn.Module):
    """
    Hierarchical Fusion Layer based on the ARISE architecture.
    Performs multi-stage hierarchical feature fusion using non-linear projection
    and dynamic gating weight estimation across within-modality and between-modality representations.
    """
    def __init__(self, dim, hidden_dim=None, act_fn=F.relu):
        super(HierarchicalFusionLayer, self).__init__()
        if hidden_dim is None:
            hidden_dim = dim
        self.dim = dim
        self.act_fn = act_fn
        
        self.fusion_fc1 = nn.Linear(2 * dim, hidden_dim, bias=True)
        self.fusion_fc2 = nn.Linear(hidden_dim, dim, bias=True)
        self.attn_fc = nn.Linear(dim, 2, bias=True)
        
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.fusion_fc1.weight)
        nn.init.zeros_(self.fusion_fc1.bias)
        nn.init.xavier_uniform_(self.fusion_fc2.weight)
        nn.init.zeros_(self.fusion_fc2.bias)
        nn.init.xavier_uniform_(self.attn_fc.weight)
        nn.init.zeros_(self.attn_fc.bias)

    def forward(self, emb1, emb2):
        combined = torch.cat([emb1, emb2], dim=-1)
        h = self.act_fn(self.fusion_fc1(combined))
        fused = self.fusion_fc2(h)
        alpha = F.softmax(self.attn_fc(fused), dim=-1)
        return fused, alpha

class ContrastiveFusionLayer(nn.Module):
    """
    Contrastive Alignment Fusion Layer.
    Projects representations into a shared space, computes temperature-scaled cosine 
    alignment similarity, and dynamically fuses modalities using contrastive weighting.
    """
    def __init__(self, dim, temperature=0.07):
        super(ContrastiveFusionLayer, self).__init__()
        self.dim = dim
        self.temperature = temperature
        self.proj1 = nn.Linear(dim, dim, bias=False)
        self.proj2 = nn.Linear(dim, dim, bias=False)
        self.fusion_fc = nn.Linear(2 * dim, dim, bias=True)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.proj1.weight)
        nn.init.xavier_uniform_(self.proj2.weight)
        nn.init.xavier_uniform_(self.fusion_fc.weight)
        nn.init.zeros_(self.fusion_fc.bias)

    def forward(self, emb1, emb2):
        z1 = F.normalize(self.proj1(emb1), p=2, dim=-1)
        z2 = F.normalize(self.proj2(emb2), p=2, dim=-1)
        
        cos_sim = (z1 * z2).sum(dim=-1, keepdim=True) / self.temperature
        w1 = torch.sigmoid(cos_sim)
        w2 = 1.0 - w1
        alpha = torch.cat([w1, w2], dim=-1)
        
        aligned1 = w1 * emb1
        aligned2 = w2 * emb2
        fused = self.fusion_fc(torch.cat([aligned1, aligned2], dim=-1))
        return fused, alpha

class GraphTransformerFusionLayer(nn.Module):
    """
    Graph Transformer Fusion Layer.
    Combines message passing with structural graph attention and positional/edge bias (b_ij),
    modeling both local neighborhood and global structural dependencies across modalities.
    
    Equations:
        alpha_ij = Softmax( (Q_i * K_j^T / sqrt(d)) + b_ij )
        Z_i = sum_j ( alpha_ij * V_j )
    """
    def __init__(self, dim):
        super(GraphTransformerFusionLayer, self).__init__()
        self.dim = dim
        self.scale = dim ** -0.5
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.bias_proj = nn.Linear(2 * dim, 1, bias=True)
        self.fc_out = nn.Linear(dim, dim, bias=True)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.q_proj.weight)
        nn.init.xavier_uniform_(self.k_proj.weight)
        nn.init.xavier_uniform_(self.v_proj.weight)
        nn.init.xavier_uniform_(self.bias_proj.weight)
        nn.init.zeros_(self.bias_proj.bias)
        nn.init.xavier_uniform_(self.fc_out.weight)
        nn.init.zeros_(self.fc_out.bias)

    def forward(self, emb1, emb2):
        q = self.q_proj(emb1)
        k = self.k_proj(emb2)
        v = self.v_proj(emb2)
        
        # Structural edge/positional bias term b_ij
        b_ij = self.bias_proj(torch.cat([emb1, emb2], dim=-1))
        
        # Graph Transformer attention score: alpha_ij = Softmax((Q_i K_j^T / sqrt(d)) + b_ij)
        scores = (q * k).sum(dim=-1, keepdim=True) * self.scale + b_ij
        alpha_val = torch.sigmoid(scores)
        alpha = torch.cat([alpha_val, 1.0 - alpha_val], dim=-1)
        
        z = alpha_val * v + (1.0 - alpha_val) * emb1
        fused = self.fc_out(z)
        return fused, alpha

class MoEFusionLayer(nn.Module):
    """
    Mixture of Experts (MoE) Fusion Layer.
    Employs multiple specialized expert fusion networks along with a learned router network 
    that dynamically weights and combines expert outputs per sample.
    
    Equations:
        g_i = Softmax(W * x)
        Z = sum_i ( g_i * E_i(x) )
    """
    def __init__(self, dim, num_experts=4):
        super(MoEFusionLayer, self).__init__()
        self.dim = dim
        self.num_experts = num_experts
        
        # Expert networks: diverse structural fusion experts
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(2 * dim, dim), nn.ReLU(), nn.Linear(dim, dim)),
            nn.Sequential(nn.Linear(2 * dim, dim), nn.GELU(), nn.Linear(dim, dim)),
            nn.Sequential(nn.Linear(2 * dim, dim), nn.Sigmoid(), nn.Linear(dim, dim)),
            nn.Sequential(nn.Linear(2 * dim, dim), nn.Tanh(), nn.Linear(dim, dim))
        ])
        
        # Router network: g_i = Softmax(W * x)
        self.router = nn.Linear(2 * dim, num_experts, bias=True)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.router.weight)
        nn.init.zeros_(self.router.bias)

    def forward(self, emb1, emb2):
        x = torch.cat([emb1, emb2], dim=-1)
        
        # Router gating weights: g_i = Softmax(W * x)
        routing_weights = F.softmax(self.router(x), dim=-1)
        
        # Compute expert outputs: Z = sum_i ( g_i * E_i(x) )
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=1)
        fused = (routing_weights.unsqueeze(-1) * expert_outputs).sum(dim=1)
        
        return fused, routing_weights

class HypergraphFusionLayer(nn.Module):
    """
    Hypergraph Fusion Layer.
    Models higher-order biological relationships (pathways, cell neighborhoods, multi-node interactions)
    using hypergraph incidence matrix convolution and non-linear feature transformation.
    
    Equation:
        H^{(l+1)} = sigma( D_v^{-1/2} H W D_e^{-1} H^T D_v^{-1/2} H^{(l)} Theta )
    """
    def __init__(self, dim, k_hyperedges=10):
        super(HypergraphFusionLayer, self).__init__()
        self.dim = dim
        self.k = k_hyperedges
        self.weight = Parameter(torch.FloatTensor(dim, dim))
        self.fusion_fc = nn.Linear(2 * dim, dim, bias=True)
        self.attn_fc = nn.Linear(dim, 2, bias=True)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight)
        nn.init.xavier_uniform_(self.fusion_fc.weight)
        nn.init.zeros_(self.fusion_fc.bias)
        nn.init.xavier_uniform_(self.attn_fc.weight)
        nn.init.zeros_(self.attn_fc.bias)

    def forward(self, emb1, emb2):
        # 1. Feature transformation: H^(l) * Theta
        h1_trans = torch.mm(emb1, self.weight)
        h2_trans = torch.mm(emb2, self.weight)
        
        # 2. Dynamic hyperedge incidence matrix construction H via top-k similarity
        z1 = F.normalize(emb1, p=2, dim=-1)
        sim_matrix = torch.mm(z1, z1.T)
        topk_vals, topk_indices = torch.topk(sim_matrix, k=min(self.k, emb1.size(0)), dim=-1)
        
        # Incidence matrix H (N x N)
        H = torch.zeros_like(sim_matrix)
        H.scatter_(1, topk_indices, 1.0)
        
        # Degree matrices: D_e (hyperedges) and D_v (nodes)
        D_e = H.sum(dim=0).clamp(min=1e-5)
        D_v = H.sum(dim=1).clamp(min=1e-5)
        
        D_v_inv_sqrt = torch.diag(torch.pow(D_v, -0.5))
        D_e_inv = torch.diag(torch.pow(D_e, -1.0))
        
        # Hypergraph convolution operator: G = D_v^{-1/2} H D_e^{-1} H^T D_v^{-1/2}
        G_hyper = torch.mm(D_v_inv_sqrt, torch.mm(H, torch.mm(D_e_inv, torch.mm(H.T, D_v_inv_sqrt))))
        
        # Convolution propagation
        hyper1 = F.relu(torch.mm(G_hyper, h1_trans))
        hyper2 = F.relu(torch.mm(G_hyper, h2_trans))
        
        combined = torch.cat([hyper1, hyper2], dim=-1)
        fused = self.fusion_fc(combined)
        alpha = F.softmax(self.attn_fc(fused), dim=-1)
        
        return fused, alpha

class SelfAttentionFusionLayer(nn.Module):
    """
    Self-Attention Fusion Layer.
    Models long-range dependencies within the fused representation space by projecting concatenated
    modalities X = [E1 || E2] into Query, Key, and Value matrices and performing intra-space self-attention.
    
    Equations:
        Q = X W_Q, K = X W_K, V = X W_V
        Z = Softmax( (Q K^T) / sqrt(d) ) V
    """
    def __init__(self, dim):
        super(SelfAttentionFusionLayer, self).__init__()
        self.dim = dim
        self.scale = dim ** -0.5
        self.q_proj = nn.Linear(2 * dim, dim, bias=False)
        self.k_proj = nn.Linear(2 * dim, dim, bias=False)
        self.v_proj = nn.Linear(2 * dim, dim, bias=False)
        self.fc_out = nn.Linear(dim, dim, bias=True)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.q_proj.weight)
        nn.init.xavier_uniform_(self.k_proj.weight)
        nn.init.xavier_uniform_(self.v_proj.weight)
        nn.init.xavier_uniform_(self.fc_out.weight)
        nn.init.zeros_(self.fc_out.bias)

    def forward(self, emb1, emb2):
        x = torch.cat([emb1, emb2], dim=-1)
        
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        
        # Intra-space self-attention: Z = Softmax( (Q K^T) / sqrt(d) ) V
        attn_scores = (q * k).sum(dim=-1, keepdim=True) * self.scale
        attn_weights = torch.sigmoid(attn_scores)
        
        z = attn_weights * v
        fused = self.fc_out(z)
        
        alpha = torch.cat([attn_weights, 1.0 - attn_weights], dim=-1)
        return fused, alpha

class ConcatFusionLayer(nn.Module):
    """
    Simple Concatenation + Linear Projection Fusion Layer (Original SpatialGLue fusion mechanism).
    Concatenates input representations along feature dimension and projects them back via a Linear layer.
    
    Equations:
        X_concat = [emb1 || emb2]
        Fused = Linear(X_concat)
    """
    def __init__(self, dim):
        super(ConcatFusionLayer, self).__init__()
        self.dim = dim
        self.fc = nn.Linear(2 * dim, dim)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, emb1, emb2):
        combined = torch.cat([emb1, emb2], dim=-1)
        fused = self.fc(combined)
        alpha = torch.ones(emb1.size(0), 2, device=emb1.device) * 0.5
        return fused, alpha

def get_fusion_layer(fusion_type: str, dim: int, attention_type: str = 'local'):
    """
    Factory function to instantiate fusion layer modules dynamically.
    Options for fusion_type: 'concat', 'gated', 'qkv', 'hierarchical', 'attention', 'contrastive', 'graph_transformer', 'moe', 'hypergraph', 'self_attention'
    """
    fusion_type = fusion_type.lower()
    if fusion_type in ['concat', 'linear', 'simple_linear', 'concat_linear']:
        return ConcatFusionLayer(dim)
    elif fusion_type == 'hierarchical':
        return HierarchicalFusionLayer(dim)
    elif fusion_type == 'qkv':
        return QKVCrossFusionLayer(dim, attention_type=attention_type)
    elif fusion_type == 'gated':
        return GatedFusionLayer(dim)
    elif fusion_type == 'attention':
        return AttentionLayer(dim, dim)
    elif fusion_type in ['contrastive', 'contrastive_alignment']:
        return ContrastiveFusionLayer(dim)
    elif fusion_type in ['graph_transformer', 'gtn', 'graphtransformer']:
        return GraphTransformerFusionLayer(dim)
    elif fusion_type in ['moe', 'mixture_of_experts']:
        return MoEFusionLayer(dim)
    elif fusion_type in ['hypergraph', 'hgn', 'hypergraph_conv']:
        return HypergraphFusionLayer(dim)
    elif fusion_type in ['self_attention', 'sa', 'self_attn']:
        return SelfAttentionFusionLayer(dim)
    else:
        raise ValueError(f"Unknown fusion type: '{fusion_type}'. Valid options are ['concat', 'gated', 'qkv', 'hierarchical', 'attention', 'contrastive', 'graph_transformer', 'moe', 'hypergraph', 'self_attention']")

class Encoder_overall(Module):
    """
    Overall encoder using PyTorch Geometric GCNConv (ARISE DualGCN architecture)
    supporting modular Within-Modality and Between-Modality fusion techniques.
    """
    def __init__(self, dim_in_feat_omics1, dim_out_feat_omics1, dim_in_feat_omics2, dim_out_feat_omics2, 
                 dropout=0.0, act=F.relu, within_fusion='graph_transformer', between_fusion='qkv', attention_type='local'):
        super(Encoder_overall, self).__init__()
        self.dim_in_feat_omics1 = dim_in_feat_omics1
        self.dim_in_feat_omics2 = dim_in_feat_omics2

        hidden_channels = dim_out_feat_omics1
        out_channels = dim_out_feat_omics1

        # RNA stream: similarity-based and distance-based branches
        self.x_RNA1 = GCNConv(self.dim_in_feat_omics1, hidden_channels)
        self.x_RNA2 = GCNConv(self.dim_in_feat_omics1, hidden_channels)

        # ADT/Protein stream
        self.protein3 = GCNConv(self.dim_in_feat_omics2, out_channels)

        self.sim_conv = GCNConv(hidden_channels, out_channels)
        self.dist_conv = GCNConv(hidden_channels, out_channels)

        # Dynamic Modular Within and Between Fusion Layers
        self.atten_within = get_fusion_layer(within_fusion, out_channels, attention_type=attention_type)
        self.atten_between = get_fusion_layer(between_fusion, out_channels, attention_type=attention_type)

        # Decoder layers for reconstruction
        self.deconv1 = nn.Linear(out_channels, hidden_channels)
        self.deconv_rna = nn.Linear(hidden_channels, self.dim_in_feat_omics1)
        self.deconv_adt = nn.Linear(hidden_channels, self.dim_in_feat_omics2)

        self.dropout = dropout

    def forward(self, features_omics1, features_omics2, sim_edge_index, sim_edge_weight, dist_edge_index, dist_edge_weight, common_edge_index, common_edge_weight):
        # Modality 1 (RNA) passes
        xs = F.relu(self.x_RNA1(features_omics1, sim_edge_index, sim_edge_weight))
        xs = F.dropout(xs, self.dropout, training=self.training)

        xd = F.relu(self.x_RNA2(features_omics1, dist_edge_index, dist_edge_weight))
        xd = F.dropout(xd, self.dropout, training=self.training)

        x_sim = self.sim_conv(xs, sim_edge_index, sim_edge_weight)
        x_dist = self.dist_conv(xd, dist_edge_index, dist_edge_weight)

        # Modality 2 (Protein/ATAC) pass
        pro = self.protein3(features_omics2, common_edge_index, common_edge_weight)

        # Within-Modality Fusion (fusing RNA similarity and distance branches)
        fused, alpha_within = self.atten_within(x_sim, x_dist)

        # Between-Modality Cross Fusion (fusing RNA fused representation and Protein/ATAC representation)
        fused_pro, alpha_between = self.atten_between(fused, pro)

        # Decode
        z_recon = F.relu(self.deconv1(fused_pro))
        emb_recon_omics1 = self.deconv_rna(z_recon)
        emb_recon_omics2 = self.deconv_adt(z_recon)

        # Consistency cross-encoding
        # Modality 1 across recon
        z_cross_1 = F.relu(self.deconv1(pro))
        rna_from_pro = self.deconv_rna(z_cross_1)
        emb_latent_omics1_across_recon = self.dist_conv(F.relu(self.x_RNA2(rna_from_pro, dist_edge_index, dist_edge_weight)), dist_edge_index, dist_edge_weight)

        # Modality 2 across recon
        z_cross_2 = F.relu(self.deconv1(fused))
        pro_from_rna = self.deconv_adt(z_cross_2)
        emb_latent_omics2_across_recon = self.protein3(pro_from_rna, common_edge_index, common_edge_weight)

        results = {
            'emb_latent_omics1': fused,
            'emb_latent_omics2': pro,
            'emb_latent_combined': fused_pro,
            'emb_recon_omics1': emb_recon_omics1,
            'emb_recon_omics2': emb_recon_omics2,
            'emb_latent_omics1_across_recon': emb_latent_omics1_across_recon,
            'emb_latent_omics2_across_recon': emb_latent_omics2_across_recon,
            'alpha_omics1': alpha_within,
            'alpha_omics2': alpha_within,
            'alpha': alpha_between
        }

        return results

def mclust_R(adata, num_cluster, modelNames='EEE', used_obsm='emb_pca', random_seed=2020):
    """\
    Clustering using the mclust algorithm.
    The parameters are the same as those in the R package mclust.
    """
    import numpy as np
    import rpy2.robjects as robjects
    from rpy2.robjects import pandas2ri
    from rpy2.robjects import default_converter
    from rpy2.robjects.conversion import localconverter
    import pandas as pd

    np.random.seed(random_seed)

    robjects.r.library("mclust")

    r_random_seed = robjects.r["set.seed"]
    r_random_seed(random_seed)

    rmclust = robjects.r["Mclust"]

    # Get the data
    X = np.array(adata.obsm[used_obsm], dtype=np.float64)

    print("Input shape:", X.shape)

    # Convert to DataFrame with column names
    df = pd.DataFrame(
        X,
        columns=[f'PC{i+1}' for i in range(X.shape[1])]
    )

    # Use subset for initialization to dramatically speed up mclust for large datasets
    subset_size = min(300, X.shape[0])
    subset_indices = robjects.IntVector(list(np.random.choice(range(1, X.shape[0] + 1), subset_size, replace=False)))
    init_list = robjects.ListVector({'subset': subset_indices})

    with localconverter(default_converter + pandas2ri.converter):
        res = rmclust(
            df,
            G=num_cluster,
            modelNames=modelNames,
            initialization=init_list
        )

    if hasattr(res, 'rx2'):
        mclust_res = np.array(res.rx2('classification'))
    elif hasattr(res, 'getbyname'):
        mclust_res = np.array(res.getbyname('classification'))
    else:
        mclust_res = np.array(res['classification'])

    adata.obs['mclust'] = mclust_res
    adata.obs['mclust'] = adata.obs['mclust'].astype('int').astype('str')
    adata.obs['mclust'] = adata.obs['mclust'].astype('category')

    return adata


def clustering(adata, n_clusters=7, key='emb', add_key='SpatialGlue', method='mclust', start=0.1, end=3.0, increment=0.01, use_pca=False, n_comps=20, random_seed=2020):
    """\
    Spatial clustering based the latent representation.

    Parameters
    ----------
    adata : anndata
        AnnData object of scanpy package.
    n_clusters : int, optional
        The number of clusters. The default is 7.
    key : string, optional
        The key of the input representation in adata.obsm. The default is 'emb'.
    method : string, optional
        The tool for clustering. Supported tools include 'mclust', 'leiden', and 'louvain'. The default is 'mclust'.
    start : float
        The start value for searching. The default is 0.1. Only works if the clustering method is 'leiden' or 'louvain'.
    end : float
        The end value for searching. The default is 3.0. Only works if the clustering method is 'leiden' or 'louvain'.
    increment : float
        The step size to increase. The default is 0.01. Only works if the clustering method is 'leiden' or 'louvain'.
    use_pca : bool, optional
        Whether use pca for dimension reduction. The default is false.
    random_seed : int, optional
        Random seed for clustering. The default is 2020.

    Returns
    -------
    None.

    """

    if use_pca:
       adata.obsm[key + '_pca'] = pca(adata, use_reps=key, n_comps=n_comps)

    if method == 'mclust':
       if use_pca:
          adata = mclust_R(adata, used_obsm=key + '_pca', num_cluster=n_clusters, random_seed=random_seed)
       else:
          adata = mclust_R(adata, used_obsm=key, num_cluster=n_clusters, random_seed=random_seed)
       adata.obs[add_key] = adata.obs['mclust']
    elif method == 'leiden':
       if use_pca:
          res = search_res(adata, n_clusters, use_rep=key + '_pca', method=method, start=start, end=end, increment=increment)
       else:
          res = search_res(adata, n_clusters, use_rep=key, method=method, start=start, end=end, increment=increment)
       sc.tl.leiden(adata, random_state=0, resolution=res)
       adata.obs[add_key] = adata.obs['leiden']
    elif method == 'louvain':
       if use_pca:
          res = search_res(adata, n_clusters, use_rep=key + '_pca', method=method, start=start, end=end, increment=increment)
       else:
          res = search_res(adata, n_clusters, use_rep=key, method=method, start=start, end=end, increment=increment)
       sc.tl.louvain(adata, random_state=0, resolution=res)
       adata.obs[add_key] = adata.obs['louvain']

def search_res(adata, n_clusters, method='leiden', use_rep='emb', start=0.1, end=3.0, increment=0.01):
    '''\
    Searching corresponding resolution according to given cluster number

    Parameters
    ----------
    adata : anndata
        AnnData object of spatial data.
    n_clusters : int
        Targetting number of clusters.
    method : string
        Tool for clustering. Supported tools include 'leiden' and 'louvain'. The default is 'leiden'.
    use_rep : string
        The indicated representation for clustering.
    start : float
        The start value for searching.
    end : float
        The end value for searching.
    increment : float
        The step size to increase.

    Returns
    -------
    res : float
        Resolution.

    '''
    print('Searching resolution...')
    label = 0
    sc.pp.neighbors(adata, n_neighbors=50, use_rep=use_rep)
    for res in sorted(list(np.arange(start, end, increment)), reverse=True):
        if method == 'leiden':
           sc.tl.leiden(adata, random_state=0, resolution=res)
           count_unique = len(pd.DataFrame(adata.obs['leiden']).leiden.unique())
           print('resolution={}, cluster number={}'.format(res, count_unique))
        elif method == 'louvain':
           sc.tl.louvain(adata, random_state=0, resolution=res)
           count_unique = len(pd.DataFrame(adata.obs['louvain']).louvain.unique())
           print('resolution={}, cluster number={}'.format(res, count_unique))
        if count_unique == n_clusters:
            label = 1
            break

    assert label==1, "Resolution is not found. Please try bigger range or smaller step!."

    return res

def plot_weight_value(alpha, label, modality1='mRNA', modality2='protein'):
  """\
  Plotting weight values

  """
  import pandas as pd

  df = pd.DataFrame(columns=[modality1, modality2, 'label'])
  df[modality1], df[modality2] = alpha[:, 0], alpha[:, 1]
  df['label'] = label
  df = df.set_index('label').stack().reset_index()
  df.columns = ['label_SpatialGlue', 'Modality', 'Weight value']
  ax = sns.violinplot(data=df, x='label_SpatialGlue', y='Weight value', hue="Modality",
                split=True, inner="quart", linewidth=1, show=False)
  ax.set_title(modality1 + ' vs ' + modality2)

  plt.tight_layout(w_pad=0.05)
  plt.show()
import torch
from tqdm import tqdm
import torch.nn.functional as F






class Train_SpatialGlue:
    def __init__(self,
        data,
        datatype = 'SPOTS',
        device= torch.device('cpu'),
        epochval=None,
        random_seed = 2022,
        learning_rate=0.0001,
        weight_decay=0.00,
        epochs=600,
        dim_input=3000,
        dim_output=64,
        weight_factors = [1, 5, 1, 1],
        within_fusion = 'graph_transformer',
        between_fusion = 'qkv',
        attention_type = 'local'
        ):
        """
        Parameters
        ----------
        data : dict
            dict object of spatial multi-omics data.
        datatype : string, optional
            Data type of input, Our current model supports 'SPOTS', 'Stereo-CITE-seq', and 'Spatial-ATAC-RNA-seq'.
        device : string, optional
            Using GPU or CPU? The default is 'cpu'.
        random_seed : int, optional
            Random seed to fix model initialization. The default is 2022.
        learning_rate : float, optional
            Learning rate for ST representation learning. The default is 0.0001.
        weight_decay : float, optional
            Weight decay to control the influence of weight parameters. The default is 0.00.
        epochs : int, optional
            Epoch for model training.
        dim_input : int, optional
            Dimension of input feature.
        dim_output : int, optional
            Dimension of output representation. The default is 64.
        weight_factors : list, optional
            Weight factors to balance the influences of different omics data.
        within_fusion : str, optional
            Within-modality fusion mechanism from scModFuse options.
        between_fusion : str, optional
            Between-modality fusion mechanism from scModFuse options.
        attention_type : str, optional
            Attention type for QKV fusion layer ('local' or 'global').
        """
        self.data = data.copy()
        self.datatype = datatype
        self.device = device
        self.random_seed = random_seed
        self.learning_rate=learning_rate
        self.weight_decay=weight_decay
        self.epochs=epochs
        self.dim_input = dim_input
        self.dim_output = dim_output
        self.weight_factors = weight_factors
        self.within_fusion = within_fusion
        self.between_fusion = between_fusion
        self.attention_type = attention_type
        self.loss_history = []

        # adj
        self.adata_omics1 = self.data['adata_omics1']
        self.adata_omics2 = self.data['adata_omics2']
        self.adj = build_dual_graph(self.adata_omics1, self.adata_omics2, device=self.device, num_neighbors=15)
        self.sim_edge_index = self.adj['sim_edge_index']
        self.sim_edge_weight = self.adj['sim_edge_weight']
        self.dist_edge_index = self.adj['dist_edge_index']
        self.dist_edge_weight = self.adj['dist_edge_weight']
        self.common_edge_index = self.adj['common_edge_index']
        self.common_edge_weight = self.adj['common_edge_weight']

        # feature
        self.features_omics1 = torch.FloatTensor(self.adata_omics1.obsm['feat'].copy()).to(self.device)
        self.features_omics2 = torch.FloatTensor(self.adata_omics2.obsm['feat'].copy()).to(self.device)

        self.n_cell_omics1 = self.adata_omics1.n_obs
        self.n_cell_omics2 = self.adata_omics2.n_obs

        # dimension of input feature
        self.dim_input1 = self.features_omics1.shape[1]
        self.dim_input2 = self.features_omics2.shape[1]
        self.dim_output1 = self.dim_output
        self.dim_output2 = self.dim_output

        print("epochval -->", epochval)

        if self.datatype == 'SPOTS':
           self.epochs = 600
           self.weight_factors = [1,5,1,1]

        elif self.datatype == 'Stereo-CITE-seq':
           self.epochs = 1500
           self.weight_factors = [1,10,1,10]

        elif self.datatype == '10x':
           self.epochs = 200
           self.weight_factors = [1,5,1,10]

        elif self.datatype == 'Spatial-epigenome-transcriptome':
           self.epochs = 1600
           self.weight_factors = [1,5,1,1]

        if epochval is not None:
            self.epochs = epochval

    def train(self):
        self.model = Encoder_overall(
            self.dim_input1, self.dim_output1, self.dim_input2, self.dim_output2,
            within_fusion=self.within_fusion,
            between_fusion=self.between_fusion,
            attention_type=self.attention_type
        ).to(self.device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), self.learning_rate,
                                          weight_decay=self.weight_decay)
        self.model.train()
        for epoch in tqdm(range(self.epochs)):
            self.model.train()
            results = self.model(self.features_omics1, self.features_omics2, self.sim_edge_index, self.sim_edge_weight, self.dist_edge_index, self.dist_edge_weight, self.common_edge_index, self.common_edge_weight)

            # reconstruction loss
            self.loss_recon_omics1 = F.mse_loss(self.features_omics1, results['emb_recon_omics1'])
            self.loss_recon_omics2 = F.mse_loss(self.features_omics2, results['emb_recon_omics2'])

            # correspondence loss
            self.loss_corr_omics1 = F.mse_loss(results['emb_latent_omics1'], results['emb_latent_omics1_across_recon'])
            self.loss_corr_omics2 = F.mse_loss(results['emb_latent_omics2'], results['emb_latent_omics2_across_recon'])

            loss = self.weight_factors[0]*self.loss_recon_omics1 + self.weight_factors[1]*self.loss_recon_omics2 + self.weight_factors[2]*self.loss_corr_omics1 + self.weight_factors[3]*self.loss_corr_omics2

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            if epoch == 0:
               print(f'Epoch {epoch} - Initial Loss: {loss.item():.4f}')

            self.loss_history.append(loss.item())

        print(f"Model training finished! Final Loss: {loss.item():.4f}\n")

        with torch.no_grad():
          self.model.eval()
          results = self.model(self.features_omics1, self.features_omics2, self.sim_edge_index, self.sim_edge_weight, self.dist_edge_index, self.dist_edge_weight, self.common_edge_index, self.common_edge_weight)

        emb_omics1 = F.normalize(results['emb_latent_omics1'], p=2, eps=1e-12, dim=1)
        emb_omics2 = F.normalize(results['emb_latent_omics2'], p=2, eps=1e-12, dim=1)
        emb_combined = F.normalize(results['emb_latent_combined'], p=2, eps=1e-12, dim=1)

        def _to_numpy(val):
            if isinstance(val, torch.Tensor):
                return val.detach().cpu().numpy()
            return np.array(val)

        output = {'emb_latent_omics1': emb_omics1.detach().cpu().numpy(),
                  'emb_latent_omics2': emb_omics2.detach().cpu().numpy(),
                  'SpatialGlue': emb_combined.detach().cpu().numpy(),
                  'alpha_omics1': _to_numpy(results['alpha_omics1']),
                  'alpha_omics2': _to_numpy(results['alpha_omics2']),
                  'alpha': _to_numpy(results['alpha']),
                  'loss_history': self.loss_history}

        return output

# ------------------------------ Fusion & Run Configuration ----------------------

# Select fusion method for within-modality (intra-omics) and between-modality (cross-omics)
# Options for WITHIN_FUSION & BETWEEN_FUSION: 'concat', 'gated', 'qkv', 'hierarchical', 'attention', 'contrastive', 'graph_transformer', 'moe', 'hypergraph', 'self_attention'

# Global default fusion technique settings
WITHIN_FUSION = 'graph_transformer'
BETWEEN_FUSION = 'qkv'

# Dataset-specific fusion technique settings (Lymph Node vs Mouse Brain)
lymphNode_WITHIN_FUSION = 'graph_transformer'
lymphNode_BETWEEN_FUSION = 'qkv'

MouseBrain_WITHIN_FUSION = 'graph_transformer'
MouseBrain_BETWEEN_FUSION = 'qkv'

choices = [
    ("10x_human_lymph_node_A1",
     "https://drive.google.com/drive/folders/10z1N4MwW8Y49o8GlkYGBKVx1N7fiMuyC"),
    ("10x_human_lymph_node_D1",
     "https://drive.google.com/drive/folders/1-g_Ca2XMaMXF-MisuVY-wobWDX86O6zz"),
    ("Mouse_Brain_E11_S1",
     "https://drive.google.com/drive/folders/1zRwDJrYnks0LRzlAVRqPU7jE_OcStgPo"),
    ("Mouse_Brain_E13_S1",
     "https://drive.google.com/drive/folders/1GOufwIRjjfcd9Bi2GKtebzKoPCg2jVud"),
    ("Mouse_Brain_E15_S1",
     "https://drive.google.com/drive/folders/1rHkTL5OF5qPsEERypRGMS51SjUQ69tdD"),
    ("Mouse_Brain_E18_S1",
     "https://drive.google.com/drive/folders/1Xj1LNIAY93biS6JIMKNRODn5GvtCKADB"),
]

# Run all 6 datasets (indices 0 to 5)
DATASET_INDICES = [0, 1, 2, 3, 4, 5]

# Define seeds to loop over
SEEDS = [
    42, 0, 1, 7, 123, 1234, 2022, 2023, 2024, 1337
]

# Set R_HOME and path before any rpy2 operations
import os
os.environ['R_HOME'] = '/usr/lib/R'
os.environ['PATH'] = '/usr/lib/R/bin:' + os.environ['PATH']

# Import rpy2 and setup mclust
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri, default_converter
import rpy2.robjects.conversion as cv

cv.set_conversion(default_converter + pandas2ri.converter)

# Suppress warnings
robjects.r.options(warn=-1)

# Install mclust only if it is not already installed
robjects.r("""
if (!requireNamespace("mclust", quietly = TRUE)) {
    install.packages("mclust", repos="https://cloud.r-project.org")
}
library(mclust)
""")

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    adjusted_mutual_info_score,
    homogeneity_score,
    v_measure_score,
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA

all_results = []
tool = 'mclust'  # mclust, leiden, and louvain
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

for dataset_idx in DATASET_INDICES:
    dataset_name, folder_url = choices[dataset_idx]

    # Dynamically select dataset-specific fusion techniques, otherwise automatically fall back to Global settings
    if "lymph_node" in dataset_name.lower() or dataset_name.startswith("10x"):
        current_within_fusion = lymphNode_WITHIN_FUSION if lymphNode_WITHIN_FUSION is not None else WITHIN_FUSION
        current_between_fusion = lymphNode_BETWEEN_FUSION if lymphNode_BETWEEN_FUSION is not None else BETWEEN_FUSION
    elif "mouse" in dataset_name.lower() or "brain" in dataset_name.lower():
        current_within_fusion = MouseBrain_WITHIN_FUSION if MouseBrain_WITHIN_FUSION is not None else WITHIN_FUSION
        current_between_fusion = MouseBrain_BETWEEN_FUSION if MouseBrain_BETWEEN_FUSION is not None else BETWEEN_FUSION
    else:
        # Automatically use Global fusion settings for any other dataset
        current_within_fusion = WITHIN_FUSION
        current_between_fusion = BETWEEN_FUSION

    print("\n" + "#"*80)
    print(f" STARTING DATASET: {dataset_name} (Within: '{current_within_fusion}', Between: '{current_between_fusion}') ".center(80, "#"))
    print("#"*80)

    # ------------------ Download & Path Configuration ------------------
    base = f"data/{dataset_name}"
    os.makedirs(base, exist_ok=True)

    rna_path = os.path.join(base, "adata_RNA.h5ad")

    # Human datasets have ADT; Mouse datasets have ATAC
    if dataset_name.startswith("10x"):
        other_path = os.path.join(base, "adata_ADT.h5ad")
        annotation_path = os.path.join(base, "annotation.csv")
        gt_column = "manual-anno"
        data_type = '10x'
    else:
        other_path = os.path.join(base, "adata_ATAC.h5ad")
        annotation_path = os.path.join(base, "anno.csv")
        gt_column = "cluster"
        data_type = 'Spatial-epigenome-transcriptome'

    # Check if files already exist locally before triggering download
    if not os.path.exists(rna_path) or not os.path.exists(other_path) or not os.path.exists(annotation_path):
        print(f"Downloading dataset files into: {base}")
        gdown_cmd = ".venv/bin/gdown" if os.path.exists(".venv/bin/gdown") else "gdown"
        os.system(f'{gdown_cmd} --folder "{folder_url}" --output "{base}"')
    else:
        print(f"Dataset files already exist at {base}. Skipping download.")

    # ------------------ Load AnnData & Annotations ------------------
    adata_omics1 = sc.read_h5ad(rna_path)
    adata_omics2 = sc.read_h5ad(other_path)

    adata_omics1.var_names_make_unique()
    adata_omics2.var_names_make_unique()

    anno_df = pd.read_csv(annotation_path, index_col=0)
    adata_omics1.obs['ground_truth'] = anno_df[gt_column]
    adata_omics2.obs['ground_truth'] = anno_df[gt_column]

    # ------------------ Preprocessing ------------------
    # RNA preprocessing
    sc.pp.filter_genes(adata_omics1, min_cells=10)
    if not dataset_name.startswith("10x"):
        sc.pp.filter_cells(adata_omics1, min_genes=200)

    sc.pp.highly_variable_genes(adata_omics1, flavor="seurat_v3", n_top_genes=3000)
    sc.pp.normalize_total(adata_omics1, target_sum=1e4)
    sc.pp.log1p(adata_omics1)
    sc.pp.scale(adata_omics1)

    adata_omics1_high = adata_omics1[:, adata_omics1.var['highly_variable']]

    # Protein or ATAC preprocessing
    if dataset_name.startswith("10x"):
        adata_omics1.obsm["feat"] = pca(
            adata_omics1_high,
            n_comps=adata_omics2.n_vars - 1
        )
        adata_omics2 = clr_normalize_each_cell(adata_omics2)
        sc.pp.scale(adata_omics2)
        adata_omics2.obsm["feat"] = pca(
            adata_omics2,
            n_comps=adata_omics2.n_vars - 1
        )
    else:
        adata_omics1.obsm["feat"] = pca(
            adata_omics1_high,
            n_comps=50
        )
        adata_omics2 = adata_omics2[adata_omics1.obs_names].copy()
        if "X_lsi" not in adata_omics2.obsm:
            sc.pp.highly_variable_genes(
                adata_omics2,
                flavor="seurat_v3",
                n_top_genes=3000
            )
            lsi(
                adata_omics2,
                use_highly_variable=False,
                n_components=51
            )
        adata_omics2.obsm["feat"] = adata_omics2.obsm["X_lsi"].copy()

    # ------------------ Spatial Neighbor Graph Construction ------------------
    data = construct_neighbor_graph(adata_omics1, adata_omics2, datatype=data_type)

    # ------------------ Seed Iterations ------------------
    dataset_results = []
    n_ground_truth = adata_omics1.obs["ground_truth"].nunique()
    print(f"Number of ground truth classes: {n_ground_truth}")
    n_cluster = n_ground_truth

    for seed in SEEDS:
        print(f"\n" + "-"*60)
        print(f" Dataset: {dataset_name} | Seed: {seed} | Within: {current_within_fusion} | Between: {current_between_fusion} ".center(60, "-"))
        print("-"*60)

        # 1. Set seed
        fix_seed(seed)

        # 2. Model Training
        import copy
        data_copy = {
            'adata_omics1': data['adata_omics1'].copy(),
            'adata_omics2': data['adata_omics2'].copy()
        }

        # Train model with dataset-specific fusion settings
        model = Train_SpatialGlue(
            data_copy,
            datatype=data_type,
            device=device,
            random_seed=seed,
            within_fusion=current_within_fusion,
            between_fusion=current_between_fusion
        )
        output = model.train()

        adata = data_copy['adata_omics1'].copy()
        adata.obsm['emb_latent_omics1'] = output['emb_latent_omics1'].copy()
        adata.obsm['emb_latent_omics2'] = output['emb_latent_omics2'].copy()
        adata.obsm['SpatialGlue'] = output['SpatialGlue'].copy()
        adata.obsm['alpha'] = output['alpha']
        adata.obsm['alpha_omics1'] = output['alpha_omics1']
        adata.obsm['alpha_omics2'] = output['alpha_omics2']

        # 3. Clustering
        clustering(adata, key='SpatialGlue', add_key='SpatialGlue', n_clusters=n_cluster, method=tool, use_pca=True, random_seed=seed)

        # 4. Score Generating
        y_true = adata.obs['ground_truth'].astype(str)
        y_pred = adata.obs['SpatialGlue'].astype(str)

        ari = adjusted_rand_score(y_true, y_pred)
        nmi = normalized_mutual_info_score(y_true, y_pred)
        ami = adjusted_mutual_info_score(y_true, y_pred)

        joint_feat = adata.obsm['SpatialGlue']
        le = LabelEncoder()
        y_pred_int = le.fit_transform(y_pred)
        sil_score = silhouette_score(joint_feat, y_pred_int)
        chi_score = calinski_harabasz_score(joint_feat, y_pred_int)
        dbi_score = davies_bouldin_score(joint_feat, y_pred_int)

        print(f"\nResult for dataset: {dataset_name} | seed: {seed}")
        print(f"ARI: {ari:.4f} | NMI: {nmi:.4f} | Silhouette: {sil_score:.4f}")

        # Store seed results
        res_dict = {
            'dataset': dataset_name,
            'seed': seed,
            'within_fusion': current_within_fusion,
            'between_fusion': current_between_fusion,
            'ARI': ari,
            'NMI': nmi,
            'AMI': ami,
            'Silhouette': sil_score,
            'CHI': chi_score,
            'DBI': dbi_score,
            'no_cluster': n_cluster
        }
        dataset_results.append(res_dict)
        all_results.append(res_dict)

    # Write dataset summary statistics and save csv for this dataset
    df_ds = pd.DataFrame(dataset_results)
    print("\n" + "="*80)
    print(f" RESULTS SUMMARY FOR {dataset_name} (Within: {current_within_fusion}, Between: {current_between_fusion}) ".center(80, "="))
    print("="*80)
    print(df_ds.to_string(index=False))
    print("-"*80)
    print("Summary Statistics:")
    print(df_ds.describe().loc[['mean', 'std']])
    print("="*80)

    # Generate text summary with Mean ± SD format for this dataset
    summary_lines = [f"Summary Stats for {dataset_name} (Within: {current_within_fusion}, Between: {current_between_fusion}):"]
    metrics_list = ['ARI', 'NMI', 'AMI', 'Silhouette', 'CHI', 'DBI']
    for metric in metrics_list:
        mean_val = df_ds[metric].mean()
        std_val = df_ds[metric].std()
        summary_lines.append(f"  {metric}: {mean_val:.4f} ± {std_val:.4f}")
    print("\n".join(summary_lines))
    print("="*80)

    # Save to CSV
    os.makedirs("results", exist_ok=True)
    df_ds.to_csv(f"results/SpatialGlue_{current_within_fusion}_{current_between_fusion}_{dataset_name}_results.csv", index=False)

# Write overall summary to file
df_all = pd.DataFrame(all_results)
all_csv_name = f"results/SpatialGlue_{WITHIN_FUSION}_{BETWEEN_FUSION}_all_results.csv" if (WITHIN_FUSION and BETWEEN_FUSION) else "results/SpatialGlue_all_results.csv"
df_all.to_csv(all_csv_name, index=False)
print("\n" + "="*80)
print(" ALL DATASETS COMPLETED ".center(80, "="))
print("="*80)
print(df_all.groupby(['dataset', 'within_fusion', 'between_fusion'])[['ARI', 'NMI', 'Silhouette']].mean())
print("="*80)